# EXP-2026-008 / Q5-E — PREP P1 + P2: 등록 자산 신원 확인

**이 노트북은 미실행 상태로 커밋된다.** 모든 출력이 비어 있고 `execution_count`
는 전부 `null` 이다.

## 무엇을 하는가

Q5-E 실행을 막고 있는 세 항목 중 **둘** 을 확정한다. P3(source-matching
differential)는 이번 범위가 **아니다**.

| | 대상 | 지금 상태 |
|---|---|---|
| **P1** | MIT-BIH publisher tree 전체 64-hex aggregate | 미등록(절단형 `0b46a411…` 만 존재) |
| **P2** | canonical Q5-D bundle 5파일 SHA-256 + subset fold | 미등록 |

## 경계 — 이 노트북이 절대 하지 않는 것

`detect_r()` 실행, M0~M4 집계, beat join 재실행, DS2 per-beat label 접근,
V10 probability 접근, association·S PR-AUC 계산, 모델 학습, Drive 파일의
이동·삭제·덮어쓰기. **파일은 바이트 단위로 해시만 하고 내용을 집계하지 않는다.**

## 이 노트북은 아무것도 등록하지 않는다

실행하면 `registration_candidates.json` 에 **관측값** 을 담을 뿐이다. 소스나
명세를 자동으로 고치지 않는다. 등록은 Codex 인수검사 뒤 **별도 결과 인수 PR**
에서만 이루어진다.

## P1 과 P2 는 독립이다

하나가 실패해도 다른 하나의 판정을 덮어쓰지 않는다. 각각의 상태와 첫 중단
사유를 따로 보존하며, **둘 다 통과할 때만** `PREP_P1_P2_PASS` 다. 실패를 규칙
완화로 해결하지 않는다.

In [ ]:
# 1. DESIGN_AND_BOUNDARIES — 셀 순서를 먼저 보여준다.
STAGES = [
    '1. DESIGN_AND_BOUNDARIES            이 표와 경계 선언',
    '2. ENVIRONMENT                      repo/모듈 로드와 staleness guard',
    '3. SYNTHETIC_FIXTURES               합성 fixture 로 경로 점검(등록 자산 미개봉)',
    '4. EXECUTION_APPROVAL               스위치 2개. 기본값은 닫힘',
    '5. DRIVE_AUTH_AND_FOLDER_ID_PREFLIGHT  folder id 로만 조회',
    '6. P1_MITDB_IDENTITY                147파일 tree 신원',
    '7. P2_Q5D_BUNDLE_IDENTITY           12파일 계약 + 5파일 input 신원',
    '8. COMBINED_DECISION                독립 gate 두 개를 합산',
    '9. BUNDLE_WRITE                     PREP bundle 원자적 기록',
    '10. HUMAN_READABLE_REPORT           사람이 읽는 요약과 다음 행동',
]
for line in STAGES:
    print(line)
print()
print('P3(source-matching differential)는 이번 범위가 아니다.')
print('이 노트북은 값을 등록하지 않는다. 관측 후보만 남긴다.')

In [ ]:
# 2. ENVIRONMENT — 모듈을 로드하고, 쓰려는 기능이 실제로 있는지 확인한다.
#    버전 정수는 안 올리면 그만이라 무력하다. 실제 capability 를 확인한다.
REPO = '/content/repo'
import os, sys, json
if not os.path.exists(REPO):
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO, 'mit-bih'))

import q5d_order_preserving_beat_join as BJ
import q5e_leg2_failure_mechanism_audit as Q5E
import q5e_prep_p1_p2_asset_identity as P

MISSING = [n for n in P.module_capabilities() if not hasattr(P, n)]
assert not MISSING, f'stale PREP clone, missing {MISSING}'
assert BJ.rule_fingerprint() == Q5E.REGISTERED_RULE_FINGERPRINT, \
    'frozen Q5-D rule fingerprint moved'

print('PREP module :', P.__file__)
print('Q5-E module :', Q5E.__file__)
print('frozen Q5-D :', BJ.__file__)
print()
print(P.design_card())

In [ ]:
# 3. SYNTHETIC_FIXTURES — 등록 자산을 열기 전에 합성 fixture 로 경로를 점검한다.
#    여기서 실패하면 실행 승인을 켜기 전에 고치는 편이 훨씬 싸다.
#    이 셀은 등록 자산을 열지 않고 Drive API 도 호출하지 않는다.
import subprocess
TEST = os.path.join(REPO, 'mit-bih', 'test_q5e_prep_p1_p2_asset_identity.py')
print(subprocess.run([sys.executable, TEST], capture_output=True,
                     text=True).stdout.strip() or '(테스트 출력 없음)')
print()
print('합성 fixture 는 FakeDriveAdapter 를 쓴다. 실제 Drive API 는 호출되지')
print('않고, 합성 결과 bundle 은 SYNTHETIC_FIXTURE 로 각인되어 ingestable=false')
print('가 된다. 즉 정식 결과로 승격될 수 없다.')

In [ ]:
# 4. EXECUTION_APPROVAL — 스위치 2개. 둘 다 켜야 등록 자산에 닿는다.
#    기본값은 닫힘이고, 이 노트북은 닫힌 상태로 커밋된다.
#    "한번 돌려보자"고 여기를 고치지 마라.
MODE_NOTE = 'read-only asset identity preflight (P1 + P2)'
APPROVAL = None                 # 실행 승인 토큰. 승인 전까지 None
OPEN_REGISTERED_DATA = False    # 등록 자산 개봉 스위치. 기본 False

print('mode                :', MODE_NOTE)
print('approval present    :', P.execution_is_approved(APPROVAL))
print('OPEN_REGISTERED_DATA:', OPEN_REGISTERED_DATA)
print()
print('둘 중 하나만 열려 있어도 거부된다. 그리고 그 뒤에 terminal guard 가')
print('한 번 더 막는다 — 이번 PR 에서는 guard 를 제거하지 않는다.')
print()
print(P.APPROVAL_NOTE)

In [ ]:
# 5. DRIVE_AUTH_AND_FOLDER_ID_PREFLIGHT
#    canonical bundle 은 folder **id** 로만 고른다. 같은 이름의 폴더를 찾은
#    것은 증거가 아니다 — 이 PREP 이 존재하는 이유가 바로 그 치환이다.
#    인증 객체는 adapter 안에만 있고 결과 bundle 에 절대 기록되지 않는다.
MITDB_DIR = ''   # 등록 publisher tree 마운트 경로 (승인 후 채운다)
MOUNT_DIR = ''   # bundle 마운트 경로. file-id 스트림이 되면 비워도 된다
OUT_DIR = '/content/drive/MyDrive/medkos_runs'

print('registered folder id :', P.SOURCE_BUNDLE_FOLDER_ID)
print('registered run       :', P.SOURCE_BUNDLE_RUN)
print('directory contract   :', len(BJ.BUNDLE_FILES), 'files')
print('Q5-E input identity  :', len(P.BUNDLE_INPUT_FILES), 'files')
print()
print('bridge 규칙: file-id 스트림이 가능하면 그것으로 끝난다(마운트 불필요).')
print('마운트를 쓴다면 folder-id inventory 와 exact name/size/count 로 연결을')
print('증명해야 하고, 증명하지 못하면 P2_FOLDER_ID_BRIDGE_UNRESOLVED 로 중단한다.')
print('폴더 이름 일치는 bridge 를 대신하지 못한다.')

In [ ]:
# 6. P1_MITDB_IDENTITY — gate 순서와 중단 규칙을 먼저 보여준다.
#    aggregate 는 앞의 세 gate 가 모두 통과한 뒤에만 계산된다. 실패한 tree 가
#    등록 후보로 오해될 숫자를 내놓지 않게 하려는 것이다.
for index, gate in enumerate(P.P1_GATE_ORDER, 1):
    print(f'  {index}. {gate}')
print()
print('expected files        :', len(BJ.mitdb_expected_files()))
print('publisher-listed      :', P.MITDB_PUBLISHER_LISTED_FILES,
      '(+1 = SHA256SUMS.txt 자체 → 147/147)')
print('checksum file digest  :', P.MITDB_CHECKSUM_FILE_SHA256)
print('registered prefix     :', P.MITDB_REGISTERED_AGGREGATE_PREFIX)
print()
print('147/147 의 뜻: publisher list 는 자기 자신을 검증할 수 없으므로')
print('146개는 목록으로, SHA256SUMS.txt 자체는 별도 등록 digest 로 확인한다.')
print('중단 사유: P1_FILE_SET_MISMATCH · P1_MITDB_CHECKSUM_FILE_MISMATCH ·')
print('P1_MITDB_PUBLISHER_CHECKSUM_MISMATCH · MITDB_IDENTITY_DIVERGED')

In [ ]:
# 7. P2_Q5D_BUNDLE_IDENTITY — 두 계약을 분리해서 확인한다.
for index, gate in enumerate(P.P2_GATE_ORDER, 1):
    print(f'  {index}. {gate}')
print()
print('A. directory contract : BJ.BUNDLE_FILES', len(BJ.BUNDLE_FILES), '개 완전성')
print('   missing/unexpected 0 · SUPERSEDED 부재 · code SHA · rule fingerprint')
print('B. input identity     : Q5-E 가 읽는', len(P.BUNDLE_INPUT_FILES), '개')
print('   각 파일 name/bytes/SHA-256 + subset fold')
print()
print('나머지', len(BJ.BUNDLE_FILES) - len(P.BUNDLE_INPUT_FILES),
      '개는 directory contract 소속이며 input identity 에서')
print('unexpected 로 취급되지 않는다. 이 혼동이 예전에 정상 bundle 을 거부했다.')
print()
print('중단 사유: P2_INVENTORY_AMBIGUOUS(중복 이름·하위 폴더·shortcut·trashed) ·')
print('P2_DIRECTORY_CONTRACT_FAILED · P2_SUPERSEDED_BUNDLE ·')
print('P2_FOLDER_ID_BRIDGE_UNRESOLVED · P2_MANIFEST_IDENTITY_MISMATCH')

In [ ]:
# 8. COMBINED_DECISION — 두 gate 는 과학적으로 독립이다.
print('가능한 종합 판정:')
for status in P.PREP_STATUSES:
    print('  ', status)
print()
print('규칙:')
print('  - 둘 다 통과해야만 PREP_P1_P2_PASS')
print('  - 하나가 실패해도 다른 하나의 판정을 덮어쓰지 않는다')
print('  - 둘 다 실패하면 MULTIPLE_PREP_FAILURES 이고 각 first_failure 를 보존')
print('  - 하나라도 실패하면 registration_allowed=false 이며,')
print('    통과한 쪽의 관측값도 등록 후보로 제시하지 않는다')

In [ ]:
# 9. BUNDLE_WRITE — 실제 실행 경로. 승인 두 개가 모두 있어야 여기까지 온다.
#    지금 상태로는 거부되며, 그것이 커밋된 의도다.
RESULT = None
if P.execution_is_approved(APPROVAL) and OPEN_REGISTERED_DATA:
    RESULT = P.run_prep(
        MITDB_DIR, P.SOURCE_BUNDLE_FOLDER_ID, OUT_DIR,
        adapter=None,              # None → 승인된 GoogleDriveFolderAdapter
        mount_dir=MOUNT_DIR or None,
        approval=APPROVAL,
        open_registered_data=OPEN_REGISTERED_DATA,
        timestamp=P.Q5E.run_timestamp() if hasattr(P, 'Q5E') else '')
    print('combined :', RESULT['combined']['status'])
    print('bundle   :', RESULT['bundle']['directory'])
else:
    print('실행하지 않았다. 위의 어떤 출력도 측정값이 아니다.')
    print()
    print('bundle 파일 계약:')
    for name in P.PREP_BUNDLE_FILES:
        print('  ', name)
    print()
    print('manifest.json 은 자기가 기록하는 payload fold 에서 제외된다.')
    print('manifest 자체 SHA-256 은 bundle 밖(Decision log·등록 기록)에서 동결한다.')
    print('bundle 안에 자기 digest 를 적으면 순환 계약이 되기 때문이다.')

In [ ]:
# 10. HUMAN_READABLE_REPORT — PASS/STOP 표와 다음 행동.
def report(result):
    if result is None:
        print('| gate | 상태 | 첫 중단 사유 |')
        print('|---|---|---|')
        print('| P1 | (미실행) | - |')
        print('| P2 | (미실행) | - |')
        print()
        print('다음 행동: 사용자 read-only 실행 승인을 받은 뒤 셀 4 의 스위치')
        print('두 개를 켜고, 셀 5 의 경로를 채운 다음 다시 실행한다.')
        return
    combined, p1, p2 = (result['combined'], result['p1'], result['p2'])
    print('| gate | 상태 | 첫 중단 사유 |')
    print('|---|---|---|')
    print(f"| P1 | {p1['status']} | {p1['first_failure']} |")
    print(f"| P2 | {p2['status']} | {p2['first_failure']} |")
    print(f"| 종합 | {combined['status']} | - |")
    print()
    print('registration_allowed :', combined['registration_allowed'])
    if combined['status'] == P.PREP_PASS:
        print()
        print('다음 행동: bundle 을 Codex 인수검사에 넘긴다. 통과하면 별도')
        print('결과 인수 PR 에서 MITDB_TREE_AGGREGATE 와')
        print('SOURCE_BUNDLE_FILE_SHA256 을 등록한다. 이 노트북은 등록하지 않는다.')
    else:
        print()
        print('다음 행동: 위 first_failure 를 그대로 보고한다. 규칙을 완화하거나')
        print('통과한 쪽만 먼저 등록하지 마라 — 둘 다 통과해야 등록이 열린다.')

report(RESULT)
print()
print('이 노트북은 Q5-E 과학 결과를 만들지 않는다. 자산 신원 확인일 뿐이며,')
print('P3(source-matching differential)는 여전히 열려 있어 Q5-E 실행은 막혀 있다.')

## 이 노트북이 해서는 안 되는 것

등록 자산 수정·이동·삭제, `detect_r()` 실행, M0~M4 집계, beat join 재실행,
DS2 per-beat label 접근, V10 probability 접근, association·S PR-AUC 계산,
모델 학습, 기존 run bundle·null shard 변경.

파일은 **바이트 해시만** 한다. parquet 을 파싱하거나 내용을 집계하지 않는다.

## 다음 단계

1. Codex 구현 인수검사
2. 사용자 read-only 실행 승인
3. P1·P2 실행 및 bundle 보존
4. Codex 결과 인수
5. 별도 결과 인수 PR 에서 값 등록
6. P3 승인·구현·실행
7. **그 뒤에만** Q5-E 실행 승인 여부를 판단한다